# Clase 1 · Fundamentos de IA Generativa y Tokenomics

## Práctica incremental

En esta notebook vas a conectar Python con Gemini, experimentar con generación y temperatura, medir tokens, estimar costos y crear el primer componente reutilizable del proyecto del curso.

Al terminar vas a tener:

- una conexión verificada con Gemini;
- mediciones de latencia y uso de tokens;
- un estimador de costos configurable;
- un cliente reutilizable guardado en `ai_agent_project/src/ai_agent_course/`;
- un reporte que será retomado en la Clase 2.

> Ejecutá las celdas en orden. Las celdas marcadas como **Actividad** requieren que escribas o modifiques algo antes de continuar.


## Definición canónica de agente

> Un agente de IA es una aplicación basada en IA que decide dinámicamente qué hacer a continuación —usando contexto, herramientas, memoria, estado y reglas de control— para avanzar hacia un objetivo bajo límites definidos.

Esta definición se mantiene alineada con las clases 1, 9 y 16. El foco está en la decisión dinámica bajo límites, no en llamar “agente” a cualquier chatbot o workflow con un LLM.

## 1. Preparar el entorno

Esta clase incorpora dos dependencias:

- `google-genai`: SDK oficial para Gemini.
- `python-dotenv`: carga segura de variables desde `.env`.

La instalación se realiza dentro del kernel activo de Jupyter.


In [1]:
# Instalación opcional de dependencias.
# En Colab o en un entorno nuevo podés cambiarlo a True y ejecutar esta celda.
# En un entorno ya preparado conviene dejarlo en False para no reinstalar paquetes en cada corrida.
RUN_INSTALLS = False

if RUN_INSTALLS:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "google-genai",
        "python-dotenv",
    ])
    print("Dependencias instaladas o actualizadas.")
else:
    print("Instalación omitida. Si falta alguna dependencia, cambiá RUN_INSTALLS a True y reejecutá la celda.")


Instalación omitida. Si falta alguna dependencia, cambiá RUN_INSTALLS a True y reejecutá la celda.


### Crear el proyecto incremental

La carpeta `ai_agent_project` se conservará durante todo el curso. Las siguientes notebooks agregarán componentes sobre esta misma base.


In [2]:
from pathlib import Path
import json
import os
import statistics
import time

PROJECT_ROOT = Path.cwd() / "ai_agent_project"
SRC_DIR = PROJECT_ROOT / "src" / "ai_agent_course"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

SRC_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").touch()

print(f"Proyecto creado en: {PROJECT_ROOT.resolve()}")


Proyecto creado en: C:\Users\CFO Tech\Documents\Proyectos\curso-ia-developer\ai-agent-developer\clase01\ai_agent_project


### Configurar la API key

Creá un archivo llamado `.env` dentro de `ai_agent_project` con este contenido:

```env
GEMINI_API_KEY=tu_api_key
GEMINI_MODEL=gemini-3.1-flash-lite
```

No muestres la clave en pantalla ni la subas a Git. La celda siguiente solo informa si fue encontrada.


In [3]:
from dotenv import load_dotenv

env_example = PROJECT_ROOT / ".env.example"
if not env_example.exists():
    env_example.write_text(
        "GEMINI_API_KEY=\nGEMINI_MODEL=gemini-3.1-flash-lite\n",
        encoding="utf-8",
    )

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
LIVE_MODE = bool(API_KEY)

print("API key detectada:", "sí" if LIVE_MODE else "no")
print("Modelo configurado:", MODEL_NAME)


API key detectada: no
Modelo configurado: gemini-3.1-flash-lite


> Si aparece `API key detectada: no`, creá o corregí `ai_agent_project/.env` y volvé a ejecutar la celda anterior. Los ejercicios conceptuales y económicos funcionan sin API, pero los experimentos de generación requieren la conexión.


## 2. Antes del código: chatbot, workflow o agente

No toda aplicación con un LLM es un agente.

- **Chatbot:** responde mensajes, pero no ejecuta un proceso propio.
- **Workflow:** recorre pasos definidos previamente por el desarrollador.
- **Agente:** decide dinámicamente qué acciones realizar para alcanzar un objetivo.
- **Sistema multiagente:** distribuye tareas entre agentes especializados.

### Actividad 1 · Clasificar casos

Completá `your_answers` usando `chatbot`, `workflow`, `agent` o `multiagent`.


In [4]:
cases = {
    "A": "Responde preguntas usando únicamente el último mensaje del usuario.",
    "B": "Resume un archivo, genera un PDF y lo envía siguiendo tres pasos fijos.",
    "C": "Decide si debe buscar documentos, calcular o pedir más información.",
    "D": "Un supervisor delega investigación y redacción a dos especialistas.",
}

your_answers = {
    "A": "",  # Completar
    "B": "",  # Completar
    "C": "",  # Completar
    "D": "",  # Completar
}

for key, description in cases.items():
    print(f"{key}. {description}\n   Tu respuesta: {your_answers[key] or 'pendiente'}\n")


A. Responde preguntas usando únicamente el último mensaje del usuario.
   Tu respuesta: pendiente

B. Resume un archivo, genera un PDF y lo envía siguiendo tres pasos fijos.
   Tu respuesta: pendiente

C. Decide si debe buscar documentos, calcular o pedir más información.
   Tu respuesta: pendiente

D. Un supervisor delega investigación y redacción a dos especialistas.
   Tu respuesta: pendiente



<details>
<summary><strong>Ver solución sugerida</strong></summary>

- A: chatbot.
- B: workflow.
- C: agente.
- D: sistema multiagente.

La diferencia central no es la interfaz: es quién decide el siguiente paso.
</details>


## 3. Primera llamada a Gemini

Creamos un cliente una sola vez. La temperatura baja favorece respuestas más estables; todavía no buscamos creatividad.


In [5]:
try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    types = None
    LIVE_MODE = False
    print(
        "No está instalado google-genai. "
        "Podés seguir las partes conceptuales; para llamar a Gemini cambiá RUN_INSTALLS a True."
    )

client = genai.Client(api_key=API_KEY) if LIVE_MODE and genai is not None else None

def require_live_mode() -> None:
    if not LIVE_MODE or client is None or types is None:
        raise RuntimeError(
            "Este experimento requiere google-genai y GEMINI_API_KEY en ai_agent_project/.env"
        )

print("Cliente preparado:", "sí" if client is not None else "modo conceptual / sin API")

print(API_KEY)

No está instalado google-genai. Podés seguir las partes conceptuales; para llamar a Gemini cambiá RUN_INSTALLS a True.
Cliente preparado: modo conceptual / sin API
None


In [6]:
if LIVE_MODE:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents="Explicá en dos oraciones qué es un modelo de lenguaje.",
        config=types.GenerateContentConfig(temperature=0.2),
    )
    print(response.text)
else:
    response = None
    print("Llamada omitida: configurá la API key para ejecutarla.")


Llamada omitida: configurá la API key para ejecutarla.


### Actividad 2 · Observar capacidades y límites

Ejecutá estos tres prompts y registrá qué puede responder el modelo por conocimiento aprendido y qué requeriría información externa actualizada.


In [7]:
diagnostic_prompts = [
    "Explicá brevemente qué es self-attention.",
    "¿Cuál es el precio exacto actual del dólar en Argentina?",
    "¿Cuántas solicitudes de vacaciones tiene pendientes mi empresa?",
]

diagnostic_results = []

if LIVE_MODE:
    for prompt in diagnostic_prompts:
        result = client.models.generate_content(model=MODEL_NAME, contents=prompt)
        diagnostic_results.append({"prompt": prompt, "response": result.text})
        print(f"PROMPT: {prompt}\nRESPUESTA: {result.text}\n{'-' * 70}")
else:
    print("Configurá la API key para ejecutar esta comparación.")


Configurá la API key para ejecutar esta comparación.


**Preguntas para responder:**

1. ¿En cuál de los casos el modelo puede explicar un concepto sin acceder a datos privados?
2. ¿En cuál necesitaría una fuente actualizada?
3. ¿En cuál necesitaría una herramienta o integración con un sistema de la empresa?

Guardá tus conclusiones en la variable siguiente.


In [8]:
capability_notes = {
    "knowledge": "",       # TODO
    "current_data": "",    # TODO
    "private_data": "",    # TODO
}
capability_notes


{'knowledge': '', 'current_data': '', 'private_data': ''}

## 4. Medir una generación

Una llamada útil para producción no devuelve solamente texto. También necesitamos observar latencia y consumo.


In [9]:
def usage_to_dict(usage) -> dict:
    if usage is None:
        return {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
    return {
        "input_tokens": int(getattr(usage, "prompt_token_count", 0) or 0),
        "output_tokens": int(getattr(usage, "candidates_token_count", 0) or 0),
        "total_tokens": int(getattr(usage, "total_token_count", 0) or 0),
    }

def measured_generation(prompt: str, temperature: float = 0.2) -> dict:
    require_live_mode()
    started = time.perf_counter()
    result = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperature),
    )
    latency_ms = round((time.perf_counter() - started) * 1000, 2)
    return {
        "prompt": prompt,
        "text": result.text or "",
        "temperature": temperature,
        "latency_ms": latency_ms,
        **usage_to_dict(result.usage_metadata),
    }

if LIVE_MODE:
    first_measurement = measured_generation(
        "Diferenciá chatbot, workflow y agente en tres viñetas."
    )
    print(json.dumps(first_measurement, indent=2, ensure_ascii=False))
else:
    first_measurement = None
    print("Medición omitida: falta la API key.")


Medición omitida: falta la API key.


### Actividad 3 · Latencia no es una constante

Ejecutá el mismo prompt cinco veces. Después compará mínimo, máximo y promedio.


In [10]:
repeated_runs = []

if LIVE_MODE:
    for _ in range(5):
        repeated_runs.append(
            measured_generation("Definí token en una sola oración.", temperature=0.0)
        )

    latencies = [run["latency_ms"] for run in repeated_runs]
    print("Mínimo:", min(latencies), "ms")
    print("Máximo:", max(latencies), "ms")
    print("Promedio:", round(statistics.mean(latencies), 2), "ms")
else:
    print("Configurá la API key para ejecutar las cinco mediciones.")


Configurá la API key para ejecutar las cinco mediciones.


## 5. Experimento de temperatura

La temperatura modifica la distribución de las respuestas. Para comparar correctamente mantenemos constantes el modelo y el prompt.

### Actividad 4 · Comparación controlada

Ejecutá tres respuestas por temperatura y observá repetición, vocabulario y variabilidad.


In [11]:
temperature_prompt = "Proponé un nombre breve para un asistente interno de políticas empresariales."
temperatures = [0.0, 0.5, 1.0]
temperature_results = []

if LIVE_MODE:
    for temperature in temperatures:
        run = measured_generation(temperature_prompt, temperature)
        temperature_results.append(run)
        print(
            f"T={temperature} | repetición=1 | "
            f"{run['text'].strip()}"
        )
else:
    print("Configurá la API key para ejecutar el experimento.")


Configurá la API key para ejecutar el experimento.


Completá una conclusión concreta. No alcanza con escribir “más temperatura = más creatividad”: describí qué observaste en estas ejecuciones.


In [12]:
temperature_conclusion = ""  # TODO: escribí tu conclusión
print(temperature_conclusion or "Conclusión pendiente")


Conclusión pendiente


## 6. Tokens y ventana de contexto

La aproximación por caracteres sirve para estimaciones rápidas, pero el conteo real depende del tokenizer del modelo.


In [13]:
def approximate_tokens(text: str) -> int:
    return max(1, round(len(text) / 4))

token_samples = [
    "Hola",
    "Explicá qué es un agente de IA.",
    "Analizá esta solicitud, buscá evidencia y prepará una respuesta con fuentes.",
]

for sample in token_samples:
    print(f"{approximate_tokens(sample):>3} tokens aproximados | {sample}")


  1 tokens aproximados | Hola
  8 tokens aproximados | Explicá qué es un agente de IA.
 19 tokens aproximados | Analizá esta solicitud, buscá evidencia y prepará una respuesta con fuentes.


### Actividad 5 · Español, inglés y longitud

Compará el conteo aproximado y el conteo real de Gemini para cada texto.


In [14]:
token_experiment = [
    {"label": "español", "text": "Explicá brevemente cómo funciona la atención."},
    {"label": "inglés", "text": "Briefly explain how attention works."},
    {"label": "conciso", "text": "Resumí el documento."},
    {
        "label": "detallado",
        "text": "Leé el documento completo, identificá sus ideas principales y escribí un resumen ejecutivo de cinco puntos.",
    },
]

for item in token_experiment:
    item["approx_tokens"] = approximate_tokens(item["text"])
    if LIVE_MODE:
        counted = client.models.count_tokens(
            model=MODEL_NAME,
            contents=item["text"],
        )
        item["model_tokens"] = int(counted.total_tokens)
    else:
        item["model_tokens"] = None

print(json.dumps(token_experiment, indent=2, ensure_ascii=False))


[
  {
    "label": "español",
    "text": "Explicá brevemente cómo funciona la atención.",
    "approx_tokens": 11,
    "model_tokens": null
  },
  {
    "label": "inglés",
    "text": "Briefly explain how attention works.",
    "approx_tokens": 9,
    "model_tokens": null
  },
  {
    "label": "conciso",
    "text": "Resumí el documento.",
    "approx_tokens": 5,
    "model_tokens": null
  },
  {
    "label": "detallado",
    "text": "Leé el documento completo, identificá sus ideas principales y escribí un resumen ejecutivo de cinco puntos.",
    "approx_tokens": 27,
    "model_tokens": null
  }
]


### Actividad 6 · Presupuesto de contexto

Suponé una ventana máxima de `1.000.000` tokens y reservá `4.000` para la salida. Calculá cuánto queda para instrucciones, historial y documentos.


In [15]:
context_window = 1_000_000
reserved_output = 4_000
system_instructions = 1_500
conversation_history = 12_000

available_for_documents = 0  # TODO: reemplazá por el cálculo

print("Tokens disponibles para documentos:", available_for_documents)


Tokens disponibles para documentos: 0


<details>
<summary><strong>Ver fórmula</strong></summary>

```python
available_for_documents = (
    context_window
    - reserved_output
    - system_instructions
    - conversation_history
)
```
</details>


## 7. Tokenomics

Las tarifas cambian con el tiempo, según el modelo y según el proveedor. En esta notebook dejamos valores de referencia para que el cálculo no quede en cero y para que el alumno pueda ver el orden de magnitud.

> Importante: antes de usar estos números para una decisión real, verificá la documentación vigente del proveedor. En esta clase usamos como default `gemini-3.1-flash-lite` en modalidad estándar paga para texto, imagen o video: USD 0,30 por millón de tokens de entrada y USD 2,50 por millón de tokens de salida. La salida incluye tokens de pensamiento cuando el modelo los consume.


In [16]:
def estimate_cost_usd(
    input_tokens: int,
    output_tokens: int,
    input_usd_per_million: float,
    output_usd_per_million: float,
) -> float:
    input_cost = input_tokens / 1_000_000 * input_usd_per_million
    output_cost = output_tokens / 1_000_000 * output_usd_per_million
    return round(input_cost + output_cost, 8)

# Precios de referencia por 1M tokens, útiles para clase y ejercicios.
# Verificá la documentación vigente antes de usar estos valores en producción.
MODEL_PRICES_USD_PER_MILLION = {
    "gemini-2.5-flash": {
        "input": 0.30,
        "output": 2.50,
        "note": "Standard paid tier, texto/imagen/video. Output incluye thinking tokens.",
    },
    "gemini-2.5-flash-lite": {
        "input": 0.10,
        "output": 0.40,
        "note": "Standard paid tier, texto/imagen/video. Output incluye thinking tokens.",
    },
    "gemini-3.1-flash-lite": {
    "input": 0.25,
    "output": 1.50,
    "note": "Standard paid tier, texto/imagen/video. Output incluye thinking tokens.",
    },
}

DEFAULT_PRICING = MODEL_PRICES_USD_PER_MILLION["gemini-3.1-flash-lite"]
MODEL_PRICING = MODEL_PRICES_USD_PER_MILLION.get(MODEL_NAME, DEFAULT_PRICING)

INPUT_USD_PER_MILLION = MODEL_PRICING["input"]
OUTPUT_USD_PER_MILLION = MODEL_PRICING["output"]

example_input_tokens = 2_500
example_output_tokens = 600

example_cost = estimate_cost_usd(
    input_tokens=example_input_tokens,
    output_tokens=example_output_tokens,
    input_usd_per_million=INPUT_USD_PER_MILLION,
    output_usd_per_million=OUTPUT_USD_PER_MILLION,
)

print("Modelo para tarifa:", MODEL_NAME if MODEL_NAME in MODEL_PRICES_USD_PER_MILLION else "gemini-3.1-flash-lite (default de referencia)")
print("Input USD / 1M tokens:", INPUT_USD_PER_MILLION)
print("Output USD / 1M tokens:", OUTPUT_USD_PER_MILLION)
print("Ejemplo tokens entrada/salida:", example_input_tokens, "/", example_output_tokens)
print("Costo estimado por llamada: USD", example_cost)


Modelo para tarifa: gemini-3.1-flash-lite
Input USD / 1M tokens: 0.25
Output USD / 1M tokens: 1.5
Ejemplo tokens entrada/salida: 2500 / 600
Costo estimado por llamada: USD 0.001525


### Actividad 7 · Del costo por llamada al costo de un agente

Un agente puede realizar varias llamadas para completar una tarea. Calculá el costo mensual del siguiente escenario y también el costo para `10.000` ejecuciones, que será parte del entregable de la clase.


In [17]:
users = 500
tasks_per_user = 8
calls_per_task = 4

avg_input_tokens_per_call = 2_500
avg_output_tokens_per_call = 600

cost_per_call = estimate_cost_usd(
    avg_input_tokens_per_call,
    avg_output_tokens_per_call,
    INPUT_USD_PER_MILLION,
    OUTPUT_USD_PER_MILLION,
)

monthly_calls = users * tasks_per_user * calls_per_task
monthly_input_tokens = monthly_calls * avg_input_tokens_per_call
monthly_output_tokens = monthly_calls * avg_output_tokens_per_call
monthly_cost = round(monthly_calls * cost_per_call, 4)

executions = 10_000
calls_for_10k_executions = executions * calls_per_task
cost_for_10k_executions = round(calls_for_10k_executions * cost_per_call, 4)

print("Costo por llamada: USD", cost_per_call)
print("Llamadas mensuales:", monthly_calls)
print("Tokens mensuales entrada:", monthly_input_tokens)
print("Tokens mensuales salida:", monthly_output_tokens)
print("Costo mensual estimado: USD", monthly_cost)
print("Costo para 10.000 ejecuciones: USD", cost_for_10k_executions)


Costo por llamada: USD 0.001525
Llamadas mensuales: 16000
Tokens mensuales entrada: 40000000
Tokens mensuales salida: 9600000
Costo mensual estimado: USD 24.4
Costo para 10.000 ejecuciones: USD 61.0


## 8. Construir el primer componente reutilizable

Hasta ahora trabajamos con celdas. Ahora llevamos la conexión, la medición y el resultado a un módulo que seguirá creciendo durante el curso.


In [18]:
client_module = 'from __future__ import annotations\n\nimport os\nimport time\nfrom dataclasses import asdict, dataclass\n\nfrom dotenv import load_dotenv\n\n\n@dataclass(frozen=True)\nclass GenerationResult:\n    text: str\n    model: str\n    temperature: float\n    latency_ms: float\n    input_tokens: int\n    output_tokens: int\n    total_tokens: int\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\nclass GeminiClient:\n    def __init__(self, env_path=None):\n        if env_path is not None:\n            load_dotenv(env_path)\n        api_key = os.getenv("GEMINI_API_KEY")\n        if not api_key:\n            raise RuntimeError("Falta GEMINI_API_KEY")\n        try:\n            from google import genai\n            from google.genai import types\n        except ImportError as exc:\n            raise RuntimeError(\n                "Falta instalar google-genai. Ejecutá: python -m pip install google-genai"\n            ) from exc\n\n        self._types = types\n        self.model = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")\n        self.client = genai.Client(api_key=api_key)\n\n    def generate(\n        self,\n        prompt: str,\n        temperature: float = 0.2,\n        max_output_tokens: int | None = None,\n        system_instruction: str | None = None,\n    ) -> GenerationResult:\n        if not prompt.strip():\n            raise ValueError("El prompt no puede estar vacío")\n        started = time.perf_counter()\n        response = self.client.models.generate_content(\n            model=self.model,\n            contents=prompt,\n            config=self._types.GenerateContentConfig(\n                temperature=temperature,\n                max_output_tokens=max_output_tokens,\n                system_instruction=system_instruction,\n            ),\n        )\n        usage = response.usage_metadata\n        return GenerationResult(\n            text=response.text or "",\n            model=self.model,\n            temperature=temperature,\n            latency_ms=round((time.perf_counter() - started) * 1000, 2),\n            input_tokens=int(getattr(usage, "prompt_token_count", 0) or 0),\n            output_tokens=int(getattr(usage, "candidates_token_count", 0) or 0),\n            total_tokens=int(getattr(usage, "total_token_count", 0) or 0),\n        )\n\n    def count_tokens(self, text: str) -> int:\n        if not text.strip():\n            return 0\n        counted = self.client.models.count_tokens(\n            model=self.model,\n            contents=text,\n        )\n        return int(counted.total_tokens)\n'

module_path = SRC_DIR / "gemini_client.py"
module_path.write_text(client_module, encoding="utf-8")
print("Módulo guardado en:", module_path)


Módulo guardado en: c:\Users\CFO Tech\Documents\Proyectos\curso-ia-developer\ai-agent-developer\clase01\ai_agent_project\src\ai_agent_course\gemini_client.py


### Probar el componente desde el proyecto


In [19]:
import sys
import importlib

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import ai_agent_course.gemini_client as gemini_module
importlib.reload(gemini_module)

if LIVE_MODE:
    project_client = gemini_module.GeminiClient(PROJECT_ROOT / ".env")
    project_result = project_client.generate(
        "Explicá qué aporta medir tokens y latencia.",
        temperature=0.2,
    )
    print(json.dumps(project_result.to_dict(), indent=2, ensure_ascii=False))
else:
    project_result = None
    print("Prueba real omitida: falta la API key.")


Prueba real omitida: falta la API key.


## 9. Desafío integrador

Creá una función `compare_prompts()` que:

1. reciba dos prompts;
2. ejecute ambos con la misma temperatura;
3. devuelva sus respuestas, tokens y latencias;
4. identifique cuál utilizó menos tokens totales.

Usala para comparar:

- `"Explicá los tokens."`
- `"Explicá qué son los tokens, cómo afectan el contexto y por qué influyen en el costo de una aplicación con LLMs."`


In [20]:
def compare_prompts(prompt_a: str, prompt_b: str, temperature: float = 0.2) -> dict:
    """Compara dos prompts usando las métricas de measured_generation().

    En modo sin API devuelve un resultado omitido para que la notebook pueda ejecutarse completa.
    """
    if not LIVE_MODE:
        return {
            "status": "omitted",
            "reason": "Configurá GEMINI_API_KEY para comparar prompts con el modelo real.",
            "prompt_a": prompt_a,
            "prompt_b": prompt_b,
        }

    runs = [
        measured_generation(prompt_a, temperature),
        measured_generation(prompt_b, temperature),
    ]
    lower_token_run = min(runs, key=lambda run: run["total_tokens"])
    faster_run = min(runs, key=lambda run: run["latency_ms"])
    return {
        "runs": runs,
        "lower_token_prompt": lower_token_run["prompt"],
        "faster_prompt": faster_run["prompt"],
    }

comparison = compare_prompts(
    "Explicá los tokens.",
    "Explicá qué son los tokens, cómo afectan el contexto y por qué influyen en el costo de una aplicación con LLMs.",
)
print(json.dumps(comparison, indent=2, ensure_ascii=False))


{
  "status": "omitted",
  "reason": "Configurá GEMINI_API_KEY para comparar prompts con el modelo real.",
  "prompt_a": "Explicá los tokens.",
  "prompt_b": "Explicá qué son los tokens, cómo afectan el contexto y por qué influyen en el costo de una aplicación con LLMs."
}


<details>
<summary><strong>Ver una implementación de referencia</strong></summary>

```python
def compare_prompts(prompt_a, prompt_b, temperature=0.2):
    if not LIVE_MODE:
        return {"status": "omitted"}

    runs = [
        measured_generation(prompt_a, temperature),
        measured_generation(prompt_b, temperature),
    ]
    winner = min(runs, key=lambda run: run["total_tokens"])
    return {
        "runs": runs,
        "lower_token_prompt": winner["prompt"],
    }
```
</details>


## 10. Guardar evidencia de la clase

El reporte no guarda la API key. Reúne configuración no sensible, mediciones y conclusiones para poder comparar cambios posteriores.


In [21]:
class_report = {
    "class": 1,
    "model": MODEL_NAME,
    "live_mode": LIVE_MODE,
    "diagnostic_results": diagnostic_results,
    "capability_notes": capability_notes,
    "first_measurement": first_measurement,
    "repeated_runs": repeated_runs,
    "temperature_results": temperature_results,
    "temperature_conclusion": temperature_conclusion,
    "token_experiment": token_experiment,
    "tokenomics": {
        "input_usd_per_million": INPUT_USD_PER_MILLION,
        "output_usd_per_million": OUTPUT_USD_PER_MILLION,
        "avg_input_tokens_per_call": avg_input_tokens_per_call,
        "avg_output_tokens_per_call": avg_output_tokens_per_call,
        "cost_per_call": cost_per_call,
        "monthly_calls": monthly_calls,
        "monthly_input_tokens": monthly_input_tokens,
        "monthly_output_tokens": monthly_output_tokens,
        "monthly_cost": monthly_cost,
        "executions": executions,
        "calls_for_10k_executions": calls_for_10k_executions,
        "cost_for_10k_executions": cost_for_10k_executions,
    },
}

report_path = ARTIFACTS_DIR / "class01_report.json"
report_path.write_text(
    json.dumps(class_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Reporte guardado en:", report_path)


Reporte guardado en: c:\Users\CFO Tech\Documents\Proyectos\curso-ia-developer\ai-agent-developer\clase01\ai_agent_project\artifacts\class01_report.json


## 11. Checkpoint final

La Clase 2 espera encontrar el proyecto, el cliente y el reporte. Esta verificación controla únicamente la estructura; las actividades `TODO` deben revisarse también.


In [22]:
checks = {
    "project_created": PROJECT_ROOT.exists(),
    "package_created": (SRC_DIR / "__init__.py").exists(),
    "gemini_client_created": (SRC_DIR / "gemini_client.py").exists(),
    "env_example_created": (PROJECT_ROOT / ".env.example").exists(),
    "report_created": (ARTIFACTS_DIR / "class01_report.json").exists(),
}

for name, passed in checks.items():
    print("✅" if passed else "❌", name)

assert all(checks.values()), "Hay componentes estructurales pendientes."
print("\nCheckpoint estructural aprobado. El proyecto está listo para la Clase 2.")


✅ project_created
✅ package_created
✅ gemini_client_created
✅ env_example_created
✅ report_created

Checkpoint estructural aprobado. El proyecto está listo para la Clase 2.


## Qué continúa en la Clase 2

La próxima notebook reutilizará `GeminiClient` y agregará:

- comparación Zero-shot y Few-shot;
- diseño de instrucciones verificables;
- Structured Outputs con Pydantic;
- validación de entradas y salidas;
- retry, fallback y evaluación por lote.

Conservá completa la carpeta `ai_agent_project`.
